In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from lstm import RNNLMScratch, LSTMScratch
device = torch.device('cpu')

Length of the dataset:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bre

In [2]:
def count_params(model):
    count = 0
    for p in model.parameters():
        count += p.numel()
    print("Number of parameters:", count)

In [11]:
with open('input.txt', 'r') as f:
    text = f.read()
print("Length of the dataset: ", len(text))

for char in "\n!$&',-.3:;?":
    if char == '\n':
        text = text.replace(char, ' ')
    else:
        text = text.replace(char, '')

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


stoi = { ch:i for i,ch in enumerate(chars)}
itos = { i:ch for i,ch in enumerate(chars)}
for ch in chars:
    assert itos[stoi[ch]] == ch
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
assert decode(encode(text[100-130])) == text[100-130]

Length of the dataset:  1115394
 ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
53


In [4]:
lstm = LSTMScratch(num_inputs=53, num_hiddens=128, sigma=0.01).to(device)
lm = RNNLMScratch(rnn=lstm, num_hiddens=128, vocab_size=53, sigma=0.01).to(device)
count_params(lm)
lm.load_state_dict(torch.load('lstm.pth', map_location=device)['state'])

Number of parameters: 100021


<All keys matched successfully>

In [12]:
print(lm.generate(' ', None, 1000, device=device))

 VI:
Your lord I wrend'st.

QUEEN MARGARET:
My lord to such find embraced end.

HERMIONE:
He is gradam, like all to in-proteediness do the claim to tell me stay.

BUCKINGHAM:
Now, the wanton of you: will you name.

SICINIUS:
I take his neck;
His lord:
O myself a marriud, employ'd: came in Lancaster,--the day very wrong bothry by undater
Stand fair and any of their poison, ever.

ABHORSON:
Mark'd, and myself commands and familist!

GLOUCESTER:
King Bason:
I love the choice of pluck hand inherians.

ISABELLA:
Everthry 'po:
Then, in Lord Marshap Meysafore:
Master; and still sulked, and the rid-gifing, he love on me:
Came attomains us follows 'twi' to their prote end: here's commiften, it is a
proof were back: 'tis a hours?
Revenged the Lady men's friend, I love a paster where she best end I see you.
What! the mind is a queen; but so I warrant wills away: whom ever
The crowns, at money; stands it be the meary bore in rifes father.

TARWICK:
All most desperate or where,
And then steak thou 

In [8]:
with open('input.txt', 'r') as f:
    text = f.read()
chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars)}
itos = { i:ch for i,ch in enumerate(chars)}
for ch in chars:
    assert itos[stoi[ch]] == ch
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
assert decode(encode(text[100-130])) == text[100-130]

lstm = LSTMScratch(num_inputs=vocab_size, num_hiddens=128, sigma=0.01).to(device)
lm = RNNLMScratch(rnn=lstm, num_hiddens=128, vocab_size=vocab_size, sigma=0.01).to(device)
count_params(lm)
lm.load_state_dict(torch.load('lstm_full_chars.pth', map_location=device)['state'])

Number of parameters: 107713


<All keys matched successfully>

In [10]:
print(lm.generate(' ', None, 2000, device))

 his dyst cousin.

KING HENRY VI:
No mourning the lapralllimate to like traitors, all say against you.

CLARENCE:
It will, see, boy? Why, whether hate you no edicapition.
Sometime to the martelo yield my maname Totenet. Foren!

RICHARD:
Gentrerably disciamation of this fly.
3 KING HENRY VI

First Lord:
Which noble with all.

STiNCES OF YORK:
Why, how no lord earth-dispapates and give my death?

Nurse:
The lark, you not.
Risham, are that point of this mind be hath made the news.

LARD Auf, to make no more. may have you again.

PRINCE:
Fould have excuseon will not mother, governs of ha
tails before King Repolcon act, no, wash'd us
Therefore is naumpost that can my courts in the atts,
Thy broken bear they say in my swattery shall be with a sew'?

CAPULET:
Bisidue sir, hearts, my brave wisely platter without humour well!

GLOUCESTER:
To make the ways doth she knew the soul to me now,
Nor at Athere KING EDWARD IV:
Not one' draw thy father; for to to shore alace the maid,
And drums call me h

In [72]:
data = torch.tensor(encode(text))

# Split train and test data
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

block_size = 8
train_data[:block_size + 1]
x = train_data[:block_size]
y = train_data[1:block_size + 1]
batch_size = 128 # how many independent sequences will be processed in parallel?
block_size = 32 # what is the maximum context length for prediction
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, ))
    x = torch.stack([data[i: i + block_size] for i in ix])
    y = torch.stack([data[i + 1: i + block_size + 1] for i in ix])
    return x, y
eval_iters = 200
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            X, Y = X.cuda(), Y.cuda()
            Y_hat, _ = model(X, None)
            losses[k] = F.cross_entropy(Y_hat.view(-1, vocab_size), Y.view(-1))
        out[split] = losses.mean()
    return out

In [73]:
estimate_loss(lm)

{'train': tensor(1.3335), 'val': tensor(1.6210)}